In [ ]:
!pip install geopy
!pip install mapillary

In [61]:
from geopy.geocoders import Nominatim
from shapely.geometry import shape, LineString
from shapely.ops import unary_union, linemerge
import requests
import os
from pathlib import Path
import math


In [65]:
geolocator = Nominatim(user_agent='streetvibes')
geolocator.geocode('shoot up hill, london').raw

{'place_id': 259407844,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'way',
 'osm_id': 1071976817,
 'lat': '51.5490063',
 'lon': '-0.2058739',
 'class': 'historic',
 'type': 'roman_road',
 'place_rank': 30,
 'importance': 9.307927061870783e-05,
 'addresstype': 'historic',
 'name': 'Shoot-up Hill',
 'display_name': 'Shoot-up Hill, Kilburn, London Borough of Camden, Greater London, England, NW2 3QL, United Kingdom',
 'boundingbox': ['51.5486916', '51.5493176', '-0.2062117', '-0.2055391']}

In [ ]:
from pyproj import Geod

def measure_length_m(line):
    geod = Geod(ellps='WGS84')
    
    if line.geom_type == 'LineString':
        lons, lats = zip(*line.coords)
        return geod.line_length(lons, lats)
    elif line.geom_type == 'MultiLineString':
        total = 0
        for ls in line.geoms:
            lons, lats = zip(*ls.coords)
            total += geod.line_length(lons, lats)
        return total
    else:
        raise ValueError("Geometry must be LineString or MultiLineString")

In [ ]:


def street_to_mapillary_images(
    query: str,
    n_points: int = 20,
    n_images_per_point: int = 1,
    point_buffer_deg: float = 0.0003,      # ~30 m search window for Mapillary
    search_radius_m: float = 2000.0,       # ~2 km radius for Overpass
    out_dir: str = 'mapillary_images'
):
    """
    1. Nominatim: locate street (name + central point)
    2. Overpass: fetch *all* highway ways with that name in a larger bbox
    3. Merge to one line, sample n_points
    4. Around each point, build a small bbox
    5. For each bbox, query Mapillary and download up to n_images_per_point images
    """

    # ---------- helpers ----------

    def point2bbox(p, buffer=point_buffer_deg):
        south, north = p.y - buffer, p.y + buffer
        west, east = p.x - buffer, p.x + buffer
        return f'{west},{south},{east},{north}'

    def get_images_from_mapillary(bbox: str, n_images: int = 1):
        token = os.environ.get('MAPILLARY_TOKEN')
        if not token:
            raise RuntimeError('MAPILLARY_TOKEN env var not set')

        url = 'https://graph.mapillary.com/images'
        params = {
            'access_token': token,
            'fields': 'id,geometry,captured_at,thumb_1024_url',
            'bbox': bbox,
            'limit': n_images,
        }

        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json().get('data', [])

        out_paths = []
        Path(out_dir).mkdir(parents=True, exist_ok=True)

        for img in data:
            img_id = img['id']
            thumb_url = img.get('thumb_1024_url')
            if not thumb_url:
                continue

            r = requests.get(thumb_url, timeout=10)
            r.raise_for_status()

            out_path = Path(out_dir) / f'{img_id}.jpg'
            with open(out_path, 'wb') as f:
                f.write(r.content)

            out_paths.append(str(out_path))

        return out_paths

    def overpass_street_lines(name: str, south: float, west: float, north: float, east: float):
        url = 'https://overpass-api.de/api/interpreter'
        query_osm = f"""
        [out:json];
        way["highway"]["name"="{name}"]({south},{west},{north},{east});
        (._; >;);
        out geom;
        """
        resp = requests.post(url, data={'data': query_osm}, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        nodes = {}
        for el in data.get('elements', []):
            if el['type'] == 'node':
                nodes[el['id']] = (float(el['lon']), float(el['lat']))

        lines = []
        for el in data.get('elements', []):
            if el['type'] == 'way':
                geom = el.get('geometry')
                if geom:
                    coords = [(g['lon'], g['lat']) for g in geom]
                else:
                    coords = [nodes[nid] for nid in el.get('nodes', []) if nid in nodes]
                if len(coords) >= 2:
                    lines.append(LineString(coords))

        return lines

    # ---------- step 1: Nominatim ----------

    geolocator = Nominatim(user_agent='streetvibes')
    loc = geolocator.geocode(query, exactly_one=True, geometry='geojson')
    if loc is None:
        raise ValueError(f'No Nominatim result for query: {query}')

    raw = loc.raw
    name = raw.get('name')
    if not name:
        raise ValueError('Nominatim result has no name field')

    # Geometry from Nominatim (for fallback + centroid)
    geom_geojson = raw.get('geojson')
    if not geom_geojson:
        raise ValueError('Nominatim result has no geojson geometry')
    nominatim_line = shape(geom_geojson)

    # Central point for bbox expansion
    lat0 = loc.latitude
    lon0 = loc.longitude

    # ---------- step 2: build a *larger* bbox for Overpass ----------

    # approximate meters -> degrees conversion
    # (good enough at city scale)
    deg_lat = search_radius_m / 110540.0
    deg_lon = search_radius_m / (111320.0 * math.cos(math.radians(lat0)) + 1e-9)

    south = lat0 - deg_lat
    north = lat0 + deg_lat
    west = lon0 - deg_lon
    east = lon0 + deg_lon

    segments = overpass_street_lines(name, south, west, north, east)

    # ---------- step 3: merge segments, fallback to Nominatim geom ----------

    if segments:
        merged = linemerge(unary_union(segments))
        if merged.geom_type == 'MultiLineString':
            line = max(merged.geoms, key=lambda g: g.length)
        else:
            line = merged
    else:
        # nothing from Overpass → fall back to Nominatim segment
        line = nominatim_line

    if n_points < 2:
        raise ValueError('n_points must be at least 2')

    fractions = [i / (n_points - 1) for i in range(n_points)]
    points = [line.interpolate(frac, normalized=True) for frac in fractions]

    print('Street length:', measure_length_m(line))
    
    # ---------- step 4: Mapillary per-point ----------

    all_paths = []
    for p in points:
        bbox_str = point2bbox(p)
        paths = get_images_from_mapillary(bbox_str, n_images=n_images_per_point)
        all_paths.extend(paths)

    return all_paths

In [ ]:
paths = street_to_mapillary_images(
    query='shoot-up hill london',
    n_points=10,
    n_images_per_point=2
)
print('\n'.join(paths))

In [ ]:
# make sure LLaVA is running: ollama run llava

import base64
import requests

def ollama_llava_image(prompt: str, image_path: str, model="llava"):
    # Read and base64-encode the image
    with open(image_path, "rb") as f:
        img_bytes = f.read()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    payload = {
        "model": model,
        "prompt": prompt,
        "images": [img_b64],
        "stream": False
    }

    resp = requests.post("http://localhost:11434/api/generate", json=payload)
    resp.raise_for_status()
    data = resp.json()

    # 'response' contains the full output when stream=False
    return data["response"]

In [ ]:
from tqdm import tqdm

In [ ]:
%%time

prompt = (
    "You are an urban planner. Describe the vibe of this street. "
    "Comment on walkability, traffic, greenery, safety, and socio-economic cues. "
    "Be concise and precise."
)

vibes = []

for path in tqdm(paths):
    
    analysis = ollama_llava_image(prompt, path)
    
    vibes.append({
        'image': path,
        'vibe': analysis
    })

In [ ]:
#vibes

In [ ]:
def build_street_summary_prompt(vibes, street_name='this street'):
    header = (
        f"You are an urban planner. You are given descriptions of ~{len(vibes)} "
        f"viewpoints along {street_name}. Each description is from an image-based "
        "model (LLaVA) and may be noisy.\n\n"
        "Tasks:\n"
        "1. Infer the overall character of the street.\n"
        "2. Comment on: land use, walkability, traffic, greenery, safety, "
        "   maintenance, socio-economic cues.\n"
        "3. Highlight any clear variation along the street (e.g. parts that feel "
        "   different).\n"
        "4. Be concise but specific. Do not repeat each segment; summarise patterns.\n\n"
        "Segment-level descriptions:\n"
    )
    body = "\n\n".join(
        f">>> Segment {i+1}: {v['vibe']}"
        for i, v in enumerate(vibes)
    )
    return header + "\n" + body

In [ ]:
%%time

import requests

def ollama_text(prompt: str, model: str = 'llama3.1'):
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
    }
    resp = requests.post('http://localhost:11434/api/generate', json=payload)
    resp.raise_for_status()
    return resp.json()['response']


prompt = build_street_summary_prompt(vibes, street_name='Station Terrace, Lampeter')
#print(prompt)
summary = ollama_text(prompt, model='llama3.1')
#print(summary)

In [ ]:
print(summary)